# Mass reconstruction, rapidity–pT efficiency and W–xV efficiency

**(a)** The same three-state mass fit, fitted yields and Gaussian-core resolutions as notebook01. **(b)** The saved 1S efficiency with true $p_{T,\Upsilon}$ horizontal and true $y_\Upsilon$ vertical. **(c)** The 1S efficiency with true $x_V$ horizontal (log scale) and true $W$ vertical. No top annotations, legends or hatching appear in (b)/(c); both colorbars read **Efficiency [%]**.

Place this notebook **beside `01_reconstruction.ipynb`**, with `settings.py` and `upsilon_esr/`. Select them once in the shared plotting profile, then **Run All**. The first run reads the original 1S ROOT file with `uproot` to build the new W–xV joint map. It validates the ROOT read against all four saved maps and caches the numerical counts. Subsequent runs reuse that cache if the source path, size, modification time, settings and saved numerical inputs agree. The ROOT path must remain accessible for this check. Set `ROOT_INPUT` only if the recorded path has moved. A full framework rerun is not needed.

The mass projection uses the shared profile luminosity (2.5 fb⁻¹ in the example) and retains the calibrated signal-only DSCB fit; resolutions are Gaussian-core widths, not RMS. Both efficiency maps use unweighted MC row counts, with the final performance selection **including the proton tag**. No luminosity scaling is applied to efficiency. The mass panel retains the existing different Q²/ID settings of 2S/3S; the maps are 1S only.

For panel (c), $x_V=(Q^2+M_{1S}^2)/(W^2+Q^2-m_p^2)$ is computed event by event. At low Q², $x_V\simeq M_{1S}^2/W^2$, so expect a narrow curved band. Gray cells have no eligible MC; they are not zero efficiency. Q²=0 and events below the old logarithmic Q² display floor are included. Display bins are not optimized cross-section bins. No interpolation or smoothing is applied.

Notebook06 and previous plots remain unchanged. Each execution writes a fresh PNG/PDF directory.

Typography is enlarged locally to match the neighboring W and differential figures at the note’s column width. `FIGURE_FONT_SCALE = 1.35` controls this display change. The central fit, counts, selections and existing shared plotting modules are unchanged.

The inset now quotes asymmetric **conditional projected statistical uncertainties**
on the fitted yields and Gaussian-core widths, using profile likelihood at the
luminosity above. The headers remain unchanged. Other free parameters are profiled;
the calibrated tails and relative widths stay fixed. The three resolution errors
come from the **same common width scale**, so their fractional errors are fully
correlated. The errors exclude background, finite-MC shape calibration and
systematic uncertainty. The nominal 68.27% profile threshold is an asymptotic
approximation, not a toy-validated coverage claim. A missing or bound-limited
interval is shown as “unc. unavailable”, never as zero. Details and full precision
are saved in the interval JSON and fit-table CSV. These are expected-histogram
projections, not errors from fitting a recorded data sample.


**Repository profile:** copy `configs/plots_9x275.example.json` to `configs/plots.local.json` and edit that one file. Paths, runs, luminosity, selections, optional `display_rebin` and diagnostic permission are shared by all notebooks. See `docs/plotting.md`. Every execution creates a fresh output directory. The current figure layouts require all three states; the 1S-only 9×130 support run is reviewed through its status/audit outputs.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import tempfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, FileLink
from settings import load_profile, make_config, new_output_dir, map_source, file_link_path
from upsilon_esr import preflight

NOTEBOOK_DIR = Path.cwd().resolve()
PROFILE = load_profile()
OUTPUT_DIR = new_output_dir(PROFILE, "mass_and_efficiency_w_xv")
cfg = make_config(profile=PROFILE, output_dir=OUTPUT_DIR)
MAP_SAMPLE = PROFILE["map_sample"]
MAP_SOURCE = map_source(PROFILE)
print("Profile:", PROFILE["profile_path"])
print("Output directory:", OUTPUT_DIR)

# null uses the original input recorded in the saved 2D selection audit.
ROOT_INPUT = PROFILE.get("root_input_override")
ROOT_TREE = PROFILE.get("root_tree", "UpsilonTree")
W_MAP_EDGES = PROFILE.get("w_map_edges")
CACHE_DIR = PROFILE["output_root"] / "map_cache" / "w_xv"
FIGURE_FONT_SCALE = 1.35  # Preserved latest note typography.


The collapsed cells retain the notebook01 mass renderer and validate the saved selections. The W–xV builder streams the ROOT input and checks all four saved maps before writing its cache.
The interval helper profiles the unchanged combined model at the plotted luminosity. It does not import the upstream MC-fit errors into this projection.


In [ ]:
def _grid(rows):
    """Rebuild the exact stored count grid, retaining undefined cells."""
    if not rows:
        raise ValueError("No numerical rows supplied for an efficiency map")
    x = np.array(sorted({float(r[k]) for r in rows for k in ("x_low", "x_high")}))
    y = np.array(sorted({float(r[k]) for r in rows for k in ("y_low", "y_high")}))
    if not (np.all(np.isfinite(x)) and np.all(np.isfinite(y))):
        raise ValueError("Nonfinite map edges")
    n = np.zeros((len(y)-1, len(x)-1), dtype=np.int64)
    k = np.zeros_like(n)
    seen = np.zeros_like(n, dtype=bool)
    for row in rows:
        ix, iy = (int(np.searchsorted(v, float(row[name])))
                  for v, name in ((x, "x_low"), (y, "y_low")))
        if (ix >= len(x)-1 or iy >= len(y)-1
                or x[ix+1] != float(row["x_high"])
                or y[iy+1] != float(row["y_high"])):
            raise ValueError("Map cells must tile adjacent edges")
        if seen[iy, ix]:
            raise ValueError("Duplicate map cell")
        ni, ki = float(row["denominator"]), float(row["selected"])
        if not (np.isfinite(ni) and np.isfinite(ki) and ni.is_integer()
                and ki.is_integer() and 0 <= ki <= ni):
            raise ValueError("Map counts require integers with 0 <= selected <= denominator")
        if ni > 0 and not np.isclose(float(row["efficiency"]), ki/ni, rtol=0, atol=1e-12):
            raise ValueError("Saved efficiency disagrees with the raw MC counts")
        n[iy,ix], k[iy,ix], seen[iy,ix] = int(ni), int(ki), True
    if not seen.all():
        raise ValueError("Incomplete stored map grid")
    values = np.divide(k, n, out=np.zeros_like(n, dtype=float), where=n > 0)
    return x, y, n, np.ma.array(100*values, mask=n == 0)


def _same_selection(left, right, name):
    a, b = np.asarray(left, dtype=float), np.asarray(right, dtype=float)
    if a.shape != b.shape or not np.isfinite(a).all() or not np.allclose(a,b,rtol=0,atol=1e-9):
        raise ValueError(f"Mass and efficiency inputs disagree on {name}: {left} versus {right}")


def validate_map_inputs(cfg, source, sample_key):
    from upsilon_esr.common import resolve_artifacts, sample_metadata
    table_path = source / "tables/final_efficiency_2d.csv"
    audit_path = source / "selection_audit.json"
    for path in (table_path, audit_path):
        if not path.is_file():
            raise FileNotFoundError(f"Missing saved 2D output: {path}. Check MAP_RUN_ID and MAP_SAMPLE.")
    audit = json.loads(audit_path.read_text())
    sample, settings = audit["config"]["sample"], audit["config"]["settings"]
    if sample["key"] != sample_key or sample["state"] != "1S":
        raise ValueError("Panels (b) and (c) require the selected 1S map sample.")
    if audit.get("status") != "COMPLETE":
        raise ValueError("Saved efficiency audit does not describe a completed calculation.")
    if settings.get("proton_selection") != "required" or audit.get("proton_selection",{}).get("policy") != "required":
        raise ValueError("The 2D maps must use the required proton tag.")
    energy = f"{settings['electron_energy']:g}x{settings['proton_energy']:g}"
    if energy != cfg.energy:
        raise ValueError("Mass and efficiency inputs have different beam energies.")
    if audit.get("weights_used") is not False or audit.get("luminosity_used") is not False:
        raise ValueError("Expected MC-count efficiencies without weights or luminosity scaling.")
    id_labels = {"new":"new electron ID", "standard":"standard electron ID"}
    if id_labels.get(sample.get("id_mode")) != cfg.categories["1S"]:
        raise ValueError("Mass and map configured electron-ID categories disagree.")
    validated = sample_metadata(cfg,"1S")
    if audit["proton_selection"].get("predicate") != validated["proton_selection"]["predicate"]:
        raise ValueError("Mass and map proton-tag predicates disagree.")
    perf_path = resolve_artifacts(cfg.runs["1S"]) / "performance" / cfg.energy / "performance_audit.json"
    perf = json.loads(perf_path.read_text())
    for key in ("Q2_GeV2", "W_GeV"):
        _same_selection(audit["truth_selection"][key],perf["truth_selection"][key],"truth "+key)
    _same_selection(audit["truth_selection"]["mass_GeV"],perf["truth_selection"]["mass_window_GeV"],"truth mass window")
    for key in ("mass_GeV", "W_GeV", "sigma_epz_GeV"):
        _same_selection(audit["reco_selection"][key],perf["reco_selection"][key],"reconstructed "+key)
    _same_selection(sample["q2_max"],cfg.q2_max["1S"],"configured Q2 maximum")
    with table_path.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows or {r.get("sample") for r in rows} != {sample_key}:
        raise ValueError("The map CSV is empty or mixes samples.")
    if {r.get("proton_selection") for r in rows} != {"required"}:
        raise ValueError("The map CSV does not consistently record proton tagging.")
    count_checks = {}
    for pkey,mkey in (("tuple_rows","raw_rows"),("eligible_truth_rows","eligible_truth"),("selected_rows","selected")):
        if pkey in perf and mkey in audit:
            if int(perf[pkey]) != int(audit[mkey]):
                raise ValueError(f"The 1S mass and map raw counts disagree: {pkey} versus {mkey}.")
            count_checks[pkey] = {"status":"PASS", "count":int(perf[pkey])}
        else:
            count_checks[pkey] = {"status":"NOT_CHECKED", "reason":"One source does not record this raw count"}
    grids = {name:_grid([r for r in rows if r["map"]==name]) for name in ("pt_y","xv_q2")}
    if np.any(grids["xv_q2"][0] <= 0) or np.any(grids["xv_q2"][1] <= 0):
        raise ValueError("The xV-Q2 map requires strictly positive log-axis edges.")
    return rows, audit, grids, {
        "selection_agreement":"PASS", "raw_count_checks":count_checks,
        "map_state":"1S", "mass_panel_states":["1S","2S","3S"],
        "sample_identity":"Matching metadata/counts do not authenticate the producer revision or prove event-by-event identity.",
        "sources":[{"path":str(p),"sha256":hashlib.sha256(p.read_bytes()).hexdigest()}
                   for p in (table_path,audit_path,perf_path)]}


def draw_combined(cfg, grids, map_audit, consistency, output_dir, font_scale=1.35):
    from dataclasses import replace
    from matplotlib.colors import Normalize
    from matplotlib.text import Text
    if not np.isfinite(font_scale) or font_scale <= 0:
        raise ValueError("FIGURE_FONT_SCALE must be finite and positive")
    from matplotlib.ticker import MaxNLocator, LogLocator, NullFormatter
    from upsilon_esr.reconstruction import plot_reconstruction
    from upsilon_esr.common import write_json
    from upsilon_esr.mass_fit_intervals import compute_mass_fit_intervals
    import pandas as pd
    import warnings
    mass_cfg = replace(cfg, output_dir=output_dir / "mass_reference")
    mass_cfg.synthetic = bool(cfg.synthetic or map_audit.get("synthetic",False))
    # Reuse notebook01's exact fit and its existing top-panel artists.
    fig, mass_audit = plot_reconstruction(mass_cfg, rebin=2,
        float_common_shape=True, efficiency_variables=("true_W","true_pT"), show_mass_intervals=False)
    # Calculate uncertainties from exactly the expected histogram and calibrated
    # signal model used by the unchanged reference fit at cfg.lumi_invfb.
    saved_mass = pd.read_csv(mass_cfg.output_dir / "upsilon_es_reconstruction_mass.csv")
    fit_edges = np.r_[saved_mass.low.to_numpy(float), float(saved_mass.high.iloc[-1])]
    fit_means = {s:saved_mass["expected_"+s].to_numpy(float) for s in ("1S","2S","3S")}
    calibrations = {s:json.loads(next(Path(p) for p in mass_audit["input_files"][s]
                         if Path(p).name == "mass_fit.json").read_text()) for s in fit_means}
    intervals = compute_mass_fit_intervals(fit_edges,fit_means,calibrations,mass_audit,
        luminosity_invfb=float(cfg.lumi_invfb))
    print("Mass table: conditional projected profile-likelihood errors at",cfg.lumi_invfb,"fb^-1")
    print("Nominal 68.27% threshold; coverage not toy-validated. Width errors share one fitted scale.")
    if any(v.get("upstream_shape_mismatch_flag",False) for v in mass_audit["states"].values()):
        warnings.warn("Upstream mass-shape mismatch is flagged; the displayed mass errors remain conditional model estimates.")
    def interval_text(entry):
        value=float(entry["value"])
        if entry["status"] != "TWO_SIDED":
            warnings.warn("Mass-table interval unavailable: "+entry["status"])
            return f"{value:.1f}\nunc. unavailable"
        return rf"${value:.1f}^{{+{entry['error_high']:.1f}}}_{{-{entry['error_low']:.1f}}}$"
    table_rows=[]
    for state in fit_means:
        row={"state":state,"luminosity_invfb":float(cfg.lumi_invfb),
             "interpretation":"conditional_projected_profile_likelihood_asymptotic"}
        for name,entry in intervals["states"][state].items():
            if name not in ("fitted_yield","fitted_sigma_core_MeV"):
                continue
            row.update({name+"_"+key:entry.get(key)
                        for key in ("value","lower","upper","error_low","error_high","status")})
        table_rows.append(row)
    pd.DataFrame(table_rows).to_csv(output_dir / "mass_fit_table.csv",index=False)
    write_json(output_dir / "mass_fit_intervals.json",intervals)
    mass_ax = fig.axes[0]
    for old_axis in list(fig.axes[1:]):
        old_axis.remove()
    fig.set_size_inches(9.6, 8.8)
    layout = fig.add_gridspec(2,2, height_ratios=(1.12,1),
        left=.10,right=.93,bottom=.09,top=.96,hspace=.46,wspace=.62)
    mass_ax.set_subplotspec(layout[0,:])
    # Keep the mass curve, table and legend intact; align the panel identifier.
    for text in mass_ax.texts:
        if text.get_text()=="(a)":
            text.set_position((0,1.035));text.set_ha("left");text.set_va("bottom")
    # Give the larger mass legend and uncertainty table more room.
    mass_ax.get_legend().set_bbox_to_anchor((.025, .67))
    # Keep the enlarged annotation above even a narrow smooth mass peak.
    peaks=[float(np.max(line.get_ydata())) for line in mass_ax.lines if len(line.get_ydata())]
    peaks += [float(np.max(patch.get_path().vertices[:,1])) for patch in mass_ax.patches]
    mass_ax.set_ylim(0,max(mass_ax.get_ylim()[1],1.85*max(peaks)))
    mass_ax.set_ylabel(mass_ax.get_ylabel().replace(" / ","\n/ ",1))
    old_table = mass_ax.tables[0]
    old_cells = old_table.get_celld()
    header = [old_cells[0,j].get_text().get_text() for j in range(3)]
    body = [[old_cells[i,0].get_text().get_text(),
             interval_text(intervals["states"][state]["fitted_yield"]),
             interval_text(intervals["states"][state]["fitted_sigma_core_MeV"])]
            for i,state in enumerate(("1S","2S","3S"),start=1)]
    table_colors = {key:cell.get_text().get_color() for key,cell in old_cells.items()}
    table_font = old_cells[1,0].get_text().get_fontsize()
    old_table.remove()
    table = mass_ax.table(cellText=body,colLabels=header,colWidths=[.22,.38,.40],
        cellLoc="center",colLoc="center",bbox=[.53,.49,.45,.41],edges="open")
    table.auto_set_font_size(False)
    table.set_fontsize(table_font)
    for (row,col),cell in table.get_celld().items():
        cell.PAD=.025
        cell.set_text_props(color=table_colors[row,col])
        if row==0:
            cell.set_text_props(weight="bold")
            cell.visible_edges="B"
            cell.set_edgecolor("#777777")
            cell.set_linewidth(.6)
    definitions = [
        ("pt_y", "b", r"$p_{T,\Upsilon}^{\mathrm{true}}$ [GeV]",r"$y_{\Upsilon}^{\mathrm{true}}$",False),
        ("xv_w", "c", r"$x_V^{\mathrm{true}}$",r"$W_{\mathrm{true}}$ [GeV]",True),
    ]
    map_axes=[]
    for column,(name,panel,xlabel,ylabel,logarithmic) in enumerate(definitions):
        inner=layout[1,column].subgridspec(1,2,width_ratios=(1,.055),wspace=.10)
        ax=fig.add_subplot(inner[0,0]);cax=fig.add_subplot(inner[0,1])
        x,y,n,efficiency=grids[name]
        cmap=plt.get_cmap("viridis").copy();cmap.set_bad("#eeeeee")
        mesh=ax.pcolormesh(x,y,efficiency,cmap=cmap,norm=Normalize(0,100),
                           shading="flat",rasterized=True)
        if logarithmic:
            ax.set_xscale("log")
            ax.xaxis.set_major_locator(LogLocator(base=10,numticks=4))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.xaxis.set_minor_formatter(NullFormatter())
        else:
            ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.set_xlim(x[0],x[-1]);ax.set_ylim(y[0],y[-1])
        ax.set_xlabel(xlabel,fontsize=15,labelpad=7)
        ax.set_ylabel(ylabel,fontsize=15,labelpad=7)
        ax.tick_params(which="both",labelsize=12.5,top=True,right=True,direction="in")
        # No campaign, state, energy or Q2 headings in the lower panels.
        ax.text(0,1.035,f"({panel})",transform=ax.transAxes,ha="left",va="bottom",fontsize=15)
        colorbar=fig.colorbar(mesh,cax=cax,ticks=np.arange(0,101,20))
        colorbar.set_label("Efficiency [%]",fontsize=13,labelpad=8)
        colorbar.ax.tick_params(labelsize=11.5,direction="in")
        map_axes.append(ax)
    # Realize ticks before scaling all figure text once, including table cells.
    # Do not change global rcParams or FONT_SIZES used by notebooks 01–03.
    fig.canvas.draw()
    seen=set()
    for artist in fig.findobj(match=Text):
        if id(artist) not in seen:
            artist.set_fontsize(artist.get_fontsize()*font_scale)
            seen.add(id(artist))
    stem="upsilon_mass_and_efficiency_w_xv"
    paths=[]
    for ext in ("pdf","png"):
        path=output_dir/(stem+"."+ext)
        fig.savefig(path,bbox_inches="tight",pad_inches=.12,dpi=220)
        paths.append(str(path))
    metadata={
        "figure":stem,"energy":cfg.energy,"mass_luminosity_invfb":float(cfg.lumi_invfb),
        "panels":{"a":"Three-state expected mass spectrum and conditional DSCB fit",
                  "b":"1S MC efficiency, true pT horizontal and true meson rapidity vertical",
                  "c":"1S MC efficiency, true xV horizontal and true W vertical"},
        "lower_panel_top_annotations":False,"hatching":False,"colorbar":"Efficiency [%]",
        "mass_fit_method_changed":False,"map_luminosity_scaling_applied":False,
        "figure_font_scale":float(font_scale),"typography_scope":"This combined figure only",
        "mass_audit":mass_audit,"map_audit":map_audit,"input_consistency":consistency,
        "mass_audit_scope":"Unchanged notebook01 reference before uncertainty annotation",
        "displayed_mass_fit_table":table_rows,"mass_parameter_intervals":intervals,
        "mass_table_headers_changed":False,
        "additional_outputs":[str(output_dir / "mass_fit_table.csv"),str(output_dir / "mass_fit_intervals.json")],
        "w_xv_map":consistency["w_xv_map"],
        "synthetic":mass_cfg.synthetic,"outputs":paths,
    }
    # Fingerprints of mass inputs are numerical source files, never ROOT rereads.
    metadata["mass_input_fingerprints"]=[{"path":p,"sha256":hashlib.sha256(Path(p).read_bytes()).hexdigest()}
        for paths_for_state in mass_audit["input_files"].values() for p in paths_for_state]
    write_json(output_dir/(stem+"_audit.json"),metadata)
    return fig,metadata,map_axes


In [ ]:
XVW_ALGORITHM_VERSION = "xvw-final-performance-v1"
XVW_TRUTH_BRANCHES = ("true_mass","true_Q2","true_W","true_y","true_pT")
XVW_BRANCHES = XVW_TRUTH_BRANCHES + ("reco_mass","reco_W","sigma_epz","reco_topology","reco_tagged_proton_p")


def _xvw_masks(arrays, audit):
    """Same scalar predicates and all-row proton validation as the saved maps."""
    lengths=set()
    for name in XVW_BRANCHES:
        if name not in arrays:
            raise ValueError("Missing ROOT branch: "+name)
        value=np.asarray(arrays[name])
        if value.ndim!=1 or value.dtype.kind not in "iuf":
            raise ValueError("Expected scalar numeric ROOT branch: "+name)
        lengths.add(len(value))
    if len(lengths)!=1:
        raise ValueError("ROOT branches have unequal row counts")
    topology,momentum=arrays["reco_topology"],arrays["reco_tagged_proton_p"]
    if np.any(~np.isfinite(topology) | ~np.isin(topology,(0,1,2,3,4))):
        raise ValueError("Unknown, noninteger or nonfinite reconstructed proton topology")
    tagged=np.isin(topology,(1,2))
    if np.any((tagged & (~np.isfinite(momentum) | (momentum<=0))) | (~tagged & (momentum!=-999))):
        raise ValueError("Inconsistent topology/proton momentum schema")
    sample,settings=audit["config"]["sample"],audit["config"]["settings"]
    if sample["state"]!="1S" or settings["proton_selection"]!="required":
        raise ValueError("This map builder requires the tagged 1S sample")
    mass=9.46030
    lo,hi=settings["w_range"]
    ee=settings["electron_energy"]
    a=arrays
    # Retain original array precision in the selection, as in the producer.
    truth=(np.isfinite(a["true_mass"]) & (np.abs(a["true_mass"]-mass)<=settings["true_mass_window"])
           & np.isfinite(a["true_Q2"]) & (a["true_Q2"]>=0) & (a["true_Q2"]<=sample["q2_max"])
           & np.isfinite(a["true_W"]) & (a["true_W"]>=lo) & (a["true_W"]<=hi))
    pretag=(truth & np.isfinite(a["reco_mass"]) & (a["reco_mass"]>=8) & (a["reco_mass"]<=12)
            & np.isfinite(a["reco_W"]) & (a["reco_W"]>=lo) & (a["reco_W"]<=hi)
            & np.isfinite(a["sigma_epz"]) & (np.abs(a["sigma_epz"]-2*ee)<=.2*ee))
    return truth,pretag,pretag & tagged


def _xvw_histogram(x,y,xedges,yedges,mask):
    valid=np.isfinite(x) & np.isfinite(y) & (np.abs(x)<900) & (np.abs(y)<900)
    inside=mask & valid & (x>=xedges[0]) & (x<=xedges[-1]) & (y>=yedges[0]) & (y<=yedges[-1])
    counts=np.histogram2d(x[inside],y[inside],bins=(xedges,yedges))[0].T.astype(np.int64)
    return counts, {"total":int(mask.sum()),"invalid_coordinates":int(np.count_nonzero(mask & ~valid)),
                    "in_histogram":int(inside.sum()),"valid_outside_any_axis":int(np.count_nonzero(mask & valid & ~inside))}


def _xvw_wilson(k,n,coverage):
    from statistics import NormalDist
    if n==0:return None,None,None
    z=NormalDist().inv_cdf((1+coverage)/2)
    p=k/n;d=1+z*z/n
    center=(p+z*z/(2*n))/d
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return float(p),float(max(0,center-half)),float(min(1,center+half))


def build_or_load_xvw(map_rows,map_audit,consistency,cache_parent,root_override=None,
                      tree_name="UpsilonTree",w_edges=None,step_size="64 MB"):
    sample,settings=map_audit["config"]["sample"],map_audit["config"]["settings"]
    root_path=Path(root_override or sample["path"]).expanduser().resolve()
    if not root_path.is_file():
        raise FileNotFoundError(f"Panel (c) needs the original ROOT input once: {root_path}. Set ROOT_INPUT to its accessible location.")
    before=root_path.stat()
    source={"path":str(root_path),"size_bytes":before.st_size,"mtime_ns":before.st_mtime_ns}
    definitions=map_audit["x_definitions"]
    mass=float(definitions["nominal_state_mass_GeV"]);mp=float(definitions["proton_mass_GeV"])
    _same_selection(mass,9.46030,"nominal 1S mass")
    _same_selection(mp,.9382720813,"proton mass in xV")
    saved={name:_grid([r for r in map_rows if r["map"]==name])
           for name in ("pt_q2","pt_y","xv_q2","xb_q2")}
    xe=saved["xv_q2"][0]
    lo,hi=map_audit["truth_selection"]["W_GeV"]
    we=np.asarray(w_edges if w_edges is not None else np.r_[np.arange(lo,hi,5.),hi],dtype=float)
    if we.ndim!=1 or len(we)<2 or not np.isfinite(we).all() or np.any(np.diff(we)<=0):
        raise ValueError("W_MAP_EDGES must be finite and strictly increasing")
    _same_selection([we[0],we[-1]],[lo,hi],"W map coverage")
    signature={"algorithm":XVW_ALGORITHM_VERSION,"root":source,"tree":tree_name,
        "sample":sample,"settings":settings,"proton_selection":map_audit["proton_selection"],
        "xv_edges":xe.tolist(),"w_edges":we.tolist(),"source_files":consistency["sources"],
        "x_definition":definitions}
    key=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    cache=Path(cache_parent)/key
    csv_path=cache/"xv_w_efficiency.csv";audit_path=cache/"xv_w_audit.json"
    if cache.exists():
        if not csv_path.is_file() or not audit_path.is_file():
            raise ValueError(f"Incomplete map cache: {cache}. Keep it for inspection and choose another CACHE_DIR.")
        cached=json.loads(audit_path.read_text())
        if cached.get("status")!="COMPLETE" or cached.get("signature")!=signature:
            raise ValueError("Cached W-xV map has incompatible metadata")
        if cached.get("csv_sha256")!=hashlib.sha256(csv_path.read_bytes()).hexdigest():
            raise ValueError("Cached W-xV counts changed since validation")
        with csv_path.open(newline="") as stream:rows=list(csv.DictReader(stream))
        grid=_grid(rows)
        _same_selection(grid[0],xe,"cached xV edges");_same_selection(grid[1],we,"cached W edges")
        print("Cached W-xV map:",cache)
        return grid,cached,cache
    try:
        import uproot
    except ImportError as error:
        raise ImportError("Building W vs xV requires uproot. Run %pip install uproot in a separate SWAN cell, then rerun.") from error
    totals={"raw_rows":0,"eligible_truth":0,"selected_before_proton":0,"selected":0,
            "zero_q2_eligible_truth":0,"zero_q2_selected":0}
    counts={name:np.zeros((len(we)-1,len(xe)-1),dtype=np.int64)
            for name in ("denominator","selected_before_proton","selected")}
    closure={name:{field:np.zeros_like(grid[2]) for field in counts} for name,grid in saved.items()}
    flows={name:{} for name in counts}
    digests={name:hashlib.sha256() for name in XVW_BRANCHES}
    print("Building W-xV map from",root_path,flush=True)
    with uproot.open(root_path) as source_file:
        if tree_name not in source_file:raise ValueError("Missing ROOT tree "+tree_name)
        tree=source_file[tree_name]
        entries=int(tree.num_entries)
        if entries<=0:raise ValueError("Input ROOT tree is empty")
        missing=set(XVW_BRANCHES)-set(tree.keys())
        if missing:raise ValueError("Missing ROOT branches: "+", ".join(sorted(missing)))
        for chunk in tree.iterate(expressions=list(XVW_BRANCHES),step_size=step_size,library="np"):
            a=({name:chunk[name] for name in chunk.dtype.names}
               if isinstance(chunk,np.ndarray) and chunk.dtype.names else dict(chunk))
            truth,pretag,selected=_xvw_masks(a,map_audit)
            masks={"denominator":truth,"selected_before_proton":pretag,"selected":selected}
            totals["raw_rows"]+=len(truth)
            for key,mask in (("eligible_truth",truth),("selected_before_proton",pretag),("selected",selected)):
                totals[key]+=int(mask.sum())
            q2=np.asarray(a["true_Q2"],dtype=np.float64);w=np.asarray(a["true_W"],dtype=np.float64)
            totals["zero_q2_eligible_truth"]+=int(np.count_nonzero(truth & (q2==0)))
            totals["zero_q2_selected"]+=int(np.count_nonzero(selected & (q2==0)))
            with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
                denominator=w*w+q2-mp*mp
                valid=np.isfinite(denominator) & (denominator>0)
                xv=np.divide(q2+mass*mass,denominator,out=np.full(len(w),np.nan),where=valid)
                xb=np.divide(q2,denominator,out=np.full(len(w),np.nan),where=valid)
            coords={"pt_q2":(a["true_pT"],q2),"pt_y":(a["true_pT"],a["true_y"]),
                    "xv_q2":(xv,q2),"xb_q2":(xb,q2)}
            for field,mask in masks.items():
                h,flow=_xvw_histogram(xv,w,xe,we,mask);counts[field]+=h
                for name,value in flow.items():flows[field][name]=flows[field].get(name,0)+value
                for name,(x,y) in coords.items():
                    closure[name][field]+=_xvw_histogram(x,y,saved[name][0],saved[name][1],mask)[0]
            for name in XVW_BRANCHES:digests[name].update(np.asarray(a[name],dtype="<f8").tobytes())
            print("Read",totals["raw_rows"],"/",entries,flush=True)
    after=root_path.stat()
    if (after.st_size,after.st_mtime_ns)!=(before.st_size,before.st_mtime_ns):
        raise RuntimeError("ROOT input changed during the read")
    if totals["raw_rows"]!=entries:raise ValueError("Read count disagrees with ROOT entry count")
    for key,value in totals.items():
        if key in map_audit and int(map_audit[key])!=value:
            raise ValueError(f"ROOT reread disagrees with saved map {key}: {value} versus {map_audit[key]}")
    hashes={name:h.hexdigest() for name,h in digests.items()}
    stored_hashes=map_audit.get("truth_content_sha256",{})
    for name in XVW_TRUTH_BRANCHES:
        if name in stored_hashes and hashes[name]!=stored_hashes[name]:
            raise ValueError("ROOT truth changed since the saved maps: "+name)
    # Compare integer counts in every cell of every old map, including pre-tag counts.
    for name,grid in saved.items():
        ex,ey=grid[:2]
        for row in (r for r in map_rows if r["map"]==name):
            ix=int(np.searchsorted(ex,float(row["x_low"])));iy=int(np.searchsorted(ey,float(row["y_low"])))
            for field in counts:
                if int(row[field])!=int(closure[name][field][iy,ix]):
                    raise ValueError(f"ROOT reread does not reproduce saved {name}/{field} counts")
    if np.any(counts["selected"]>counts["selected_before_proton"]) or np.any(counts["selected_before_proton"]>counts["denominator"]):
        raise AssertionError("Invalid nested W-xV selection counts")
    for field,flow in flows.items():
        if flow["in_histogram"]!=int(counts[field].sum()) or flow["total"]!=sum(flow[k] for k in ("invalid_coordinates","in_histogram","valid_outside_any_axis")):
            raise AssertionError("W-xV map count/flow partition failed")
    rows=[]
    for iy,ix in np.ndindex(counts["denominator"].shape):
        n,k=(int(counts[field][iy,ix]) for field in ("denominator","selected"))
        efficiency,low,high=_xvw_wilson(k,n,float(settings["coverage"]))
        rows.append({"sample":sample["key"],"state":"1S","id_mode":sample["id_mode"],
            "proton_selection":"required","map":"xv_w","x_variable":"true_xV","y_variable":"true_W",
            "x_low":float(xe[ix]),"x_high":float(xe[ix+1]),"y_low":float(we[iy]),"y_high":float(we[iy+1]),
            "denominator":n,"selected":k,"selected_before_proton":int(counts["selected_before_proton"][iy,ix]),
            "selected_after_proton":k,"efficiency":efficiency,"wilson_low":low,"wilson_high":high})
    if not counts["denominator"].any():raise ValueError("No eligible events populate the W-xV map")
    result={"status":"COMPLETE","signature":signature,"counts":totals,"histogram_flows":flows,
        "branch_content_sha256":hashes,"saved_map_count_closure":"PASS for all four maps, all three count fields",
        "truth_hash_check":"PASS" if all(n in stored_hashes for n in XVW_TRUTH_BRANCHES) else "Available hashes checked; incomplete saved hash coverage",
        "x_definition":"(true_Q2 + nominal_1S_mass**2)/(true_W**2 + true_Q2 - proton_mass**2)",
        "zero_q2":"Included in selection and this map; no positive Q2 display threshold is applied",
        "weights_used":False,"luminosity_used":False,"synthetic":bool(map_audit.get("synthetic",False)),
        "interpretation":"Display binning for MC efficiency; not optimized projected cross-section bins"}
    cache.mkdir(parents=True,exist_ok=False)
    with csv_path.open("x",newline="") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    result["csv_sha256"]=hashlib.sha256(csv_path.read_bytes()).hexdigest()
    with audit_path.open("x") as stream:json.dump(result,stream,indent=2,allow_nan=False)
    print("Saved W-xV map:",cache)
    return _grid(rows),result,cache


In [ ]:
display(preflight(cfg,section="reconstruction"))
map_rows,map_audit,map_grids,consistency = validate_map_inputs(cfg,MAP_SOURCE,MAP_SAMPLE)
print("Mass/map 1S selection agreement:",consistency["selection_agreement"])
print("Lower panels:",MAP_SAMPLE,"— efficiencies are not luminosity scaled")


In [ ]:
map_grids["xv_w"],wxv_audit,wxv_cache = build_or_load_xvw(
    map_rows,map_audit,consistency,CACHE_DIR,
    root_override=ROOT_INPUT,tree_name=ROOT_TREE,w_edges=W_MAP_EDGES)
consistency["w_xv_map"] = {"cache_directory":str(wxv_cache),"audit":wxv_audit}
print("ROOT/saved-map count closure:",wxv_audit["saved_map_count_closure"])
print("W–xV displayed MC counts:",wxv_audit["histogram_flows"])


In [ ]:
fig,audit,map_axes = draw_combined(cfg,map_grids,map_audit,consistency,OUTPUT_DIR,font_scale=FIGURE_FONT_SCALE)
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / "upsilon_mass_and_efficiency_w_xv.png"),width=960))


In [ ]:
print("Output directory:",OUTPUT_DIR)
for extension in ("png","pdf"):
    path=OUTPUT_DIR / ("upsilon_mass_and_efficiency_w_xv."+extension)
    display(FileLink(file_link_path(path),result_html_prefix=extension.upper()+": "))
display(FileLink(file_link_path(OUTPUT_DIR / "upsilon_mass_and_efficiency_w_xv_audit.json"),result_html_prefix="Audit: "))

for filename,label in (("mass_fit_table.csv","Fit table"),("mass_fit_intervals.json","Parameter intervals")):
    display(FileLink(file_link_path(OUTPUT_DIR / filename),result_html_prefix=label+": "))
